# ⚡ Teo v2 (~124M Parameters): Entrenamiento de Alto Rendimiento en GPU T4
### *Netelpro Neuro-Simbólico | Español + Inglés Técnico + Filosofía + Rust/C++/C#/Python*

Este notebook ejecuta el entrenamiento continuo de **Teo v2** sobre una GPU **NVIDIA T4** (16GB VRAM) aprovechando aceleración por tensores y precisión mixta (FP16 Autocast).

### 🎯 Objetivos de la sesión:
1. **Bilingüe nativo:** Fluidez en español conversacional y lectura/comprensión de documentación técnica y RFCs en inglés.
2. **Ingeniería de Sistemas:** Conocimiento profundo de Rust, C++, C#, Python y compuertas en silicio Netelpro.
3. **Throughput de alta velocidad:** ~10,000 a 14,000 tokens/segundo (procesa 250M tokens en ~5-6 horas).
4. **Checkpoint persistente:** Guardado automático con optimizador AdamW y estados para descargar o reanudar.

---
### ⚙️ Configuración en Kaggle (Panel lateral derecho):
- **Accelerator:** GPU T4 x2 (o GPU T4 x1 / P100)
- **Internet:** **On** (Necesario para clonar el repositorio y descargar el corpus)
- **Persistence:** Variables / Files on


## 1. Verificación de Hardware y Acelerador GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Dispositivo: {torch.cuda.get_device_name(0)}')
    print(f'Memoria VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')


## 2. Clonación del Repositorio e Instalación de Dependencias

In [ ]:
%cd /kaggle/working
# Clonar o actualizar repositorio Netelpro
import os
if not os.path.exists('netelpro'):
    !git clone https://github.com/jona2428/netelpro.git
else:
    !cd /kaggle/working/netelpro && git pull origin master
%cd /kaggle/working/netelpro

# Instalar requerimientos optimizados
!pip install -q tokenizers datasets pyarrow requests


## 3. Generación / Descarga del Corpus Balanceado
Construye los shards binarios empacados (Español + Inglés Técnico + Código + Filosofía + Persona Netelpro) aprovechando la conexión de fibra óptica de Kaggle (~2 Gbps).

In [ ]:
# Construir dataset balanceado multi-dominio con tokenizador congelado de 32k
!python training/data/build_balanced_dataset.py \
    --out-dir data/teo_v2_balanced \
    --tokenizer-path data/teo_v2/tokenizer.json \
    --target-shards 4 \
    --shard-size 25000000 \
    --persona-multiplier 50


## 4. Lanzamiento del Entrenamiento de Teo v2 (~124M)
Entrenamos con batch size 8 o 16 y contexto 512 en FP16 Autocast.

In [ ]:
import os
os.makedirs('models/teo_v2', exist_ok=True)

!python training/train_teo_v2.py \
    --data-dir data/teo_v2_balanced \
    --checkpoint-dir models/teo_v2 \
    --device cuda \
    --autocast \
    --batch-size 8 \
    --context-len 512 \
    --lr 3e-4 \
    --warmup-steps 1000 \
    --save-interval 500 \
    --log-interval 50


## 5. Prueba de Inferencia y Generación en Vivo con Teo

In [ ]:
import torch
from netelpro.neuro.tokenizer_bpe import NetelproBPETokenizer
from training.train_teo_v2 import TeoV2Config, TeoV2Transformer, load_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = NetelproBPETokenizer.load('data/teo_v2/tokenizer.json')
cfg = TeoV2Config.from_json('training/teo_v2_config.json')
cfg.context = 512

model = TeoV2Transformer(cfg).to(device)
load_checkpoint('models/teo_v2/checkpoint.pt', model=model, device=device)
model.eval()
print('✅ Modelo cargado exitosamente para inferencia.')

prompt = '<|user|>\nwena teo aca jona, como te sientes?<|assistant|>\n'
input_ids = torch.tensor([tok.encode(prompt)], device=device)

with torch.no_grad():
    for _ in range(120):
        logits = model(input_ids)[:, -1, :]
        next_tok = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_tok], dim=1)
        if next_tok.item() == tok.eos_token_id:
            break

response = tok.decode(input_ids[0].tolist())
print('--- RESPUESTA DE TEO ---')
print(response)


## 6. Empaquetar Checkpoint Final para Descarga a tu PC

In [ ]:
!tar -czvf /kaggle/working/teo_v2_final_checkpoint.tar.gz -C models/teo_v2 checkpoint.pt
print('🎉 Archivo teo_v2_final_checkpoint.tar.gz listo en /kaggle/working para descargar!')
